In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"
GOLD_PATH = "abfss://gold@pravdatalake.dfs.core.windows.net"
GOLD_TABLE_PATH = f"{GOLD_PATH}/dim_genmodel"
GOLD_TABLE_NAME = "vehicle_sales.gold.dim_genmodel"

In [0]:
silver_basic = spark.read.format("delta").load(f"{SILVER_PATH}/basic_table")

In [0]:
dim_automaker = spark.read.format("delta").load(f"{GOLD_PATH}/dim_automaker")

In [0]:
dim_genmodel_updates = (
    silver_basic
    .select("Genmodel_ID", "Genmodel", "Automaker_ID")
    .dropDuplicates(["Genmodel_ID"])
    .join(
        dim_automaker.select("Automaker_ID").withColumnRenamed("Automaker_ID", "Automaker_ID_check"),
        col("Automaker_ID") == col("Automaker_ID_check"),
        "left"
    )
    .withColumn("gold_updated_timestamp", current_timestamp())
    .select("Genmodel_ID", "Genmodel", "Automaker_ID", "Automaker_ID_check", "gold_updated_timestamp")
)

In [0]:
dim_genmodel_updates.display()

####Data Quality checks

In [0]:
row_count = dim_genmodel_updates.count()

In [0]:
null_key_count = dim_genmodel_updates.filter(col("Genmodel_ID").isNull()).count()

In [0]:
duplicate_key_count = dim_genmodel_updates.groupBy("Genmodel_ID").count().filter("count > 1").count()

In [0]:

unresolved_fk_count = dim_genmodel_updates.filter(col("Automaker_ID_check").isNull()).count()

In [0]:

print(f"row count: {row_count}")
print(f"null Genmodel_ID count: {null_key_count}")
print(f"duplicate Genmodel_ID count: {duplicate_key_count}")
print(f"unresolved Automaker_ID (not found in dim_automaker) count: {unresolved_fk_count}")

In [0]:
assert null_key_count == 0, "Genmodel_ID should never be null in dim_genmodel"
assert duplicate_key_count == 0, "Genmodel_ID should be unique in dim_genmodel"

In [0]:

dim_genmodel_updates = dim_genmodel_updates.drop("Automaker_ID_check")

In [0]:
if DeltaTable.isDeltaTable(spark, GOLD_TABLE_PATH):
 
    dim_genmodel_table = DeltaTable.forPath(spark, GOLD_TABLE_PATH)
 
    (dim_genmodel_table.alias("t")
        .merge(dim_genmodel_updates.alias("s"), "t.Genmodel_ID = s.Genmodel_ID")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
 
else:
 
    dim_genmodel_updates.write \
        .format("delta") \
        .mode("overwrite") \
        .save(GOLD_TABLE_PATH)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE_NAME}
    USING DELTA
    LOCATION '{GOLD_TABLE_PATH}'
""")

In [0]:
spark.sql(f"OPTIMIZE {GOLD_TABLE_NAME} ZORDER BY (Genmodel_ID)")